# Data Preprocessing Notebook

## Overview
This notebook performs data preprocessing on two datasets:
1. **BoB Data** (Bill of Business) - Contains agreement and account information
2. **Retention Data** - Contains customer retention and service case information

The notebook cleans, transforms, and merges these datasets to prepare them for analysis.

## Setup & Configuration

### Import Libraries
Load pandas for data manipulation and file I/O operations.

In [1]:
import pandas as pd

### Define File Paths
Set up directory paths and file names for raw and cleaned data files.

In [2]:
path = 'dataset_2/'
raw_bob_file = path + 'BoB.xlsx'
raw_retention_file = path + 'Retention.csv'
clean_bob_file = path + 'bob_clean.xlsx'
clean_retention_file = path + 'retention_clean.xlsx'

### Utility Function: strip_all_string_cols()
This function removes leading/trailing whitespace from all string and object-type columns in a DataFrame. This is applied to both datasets to ensure consistent data formatting.


In [3]:
def strip_all_string_cols(df):
    for col in df.columns:
        if df[col].dtype in ('str', 'object'):
            df[col] = df[col].str.strip()

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## BoB Data Processing

### Load BoB Data
Read the Excel file containing Bill of Business records into a pandas DataFrame.

In [5]:
df_bob = pd.read_excel(raw_bob_file)

### Data Quality Check - Initial
Display dataset shape and data types using `.info()` and check row count.

In [6]:
print(df_bob.shape)

df_bob.info()

(376803, 26)
<class 'pandas.DataFrame'>
RangeIndex: 376803 entries, 0 to 376802
Data columns (total 26 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   account_number        376784 non-null  str           
 1   company_sizing        374120 non-null  str           
 2   postal_code           376782 non-null  str           
 3   branch                376784 non-null  str           
 4   vat_number            376388 non-null  str           
 5   agreement_number      376784 non-null  str           
 6   agreement_start_date  376784 non-null  datetime64[us]
 7   agreement_end_date    376784 non-null  datetime64[us]
 8   renewal_type          376784 non-null  str           
 9   agreement_type        376784 non-null  str           
 10  line_of_business      376784 non-null  str           
 11  system_status         376784 non-null  str           
 12  product_bob           376793 non-null  float64       
 1

### Remove Duplicate Rows
Identify and remove duplicate records from the BoB dataset to ensure data integrity.

In [7]:
print(f'Duplicate rows: {df_bob.duplicated().sum()}')

df_bob.drop_duplicates(inplace=True)

print(f'Duplicate rows after dropping: {df_bob.duplicated().sum()}')

Duplicate rows: 177814
Duplicate rows after dropping: 0


### Clean String Columns
Strip whitespace from all string columns for consistent formatting.

In [8]:
strip_all_string_cols(df_bob)

### Extract Account Number
The account_number column contains values in format "PREFIX-ID". Extract only the ID portion after splitting by the hyphen.

In [9]:
df_bob['account_number'].sample(5)

232865    UK02-SGBA213767-L
311649    UK02-SGBA014042-L
321958    UK02-SGBA024945-L
25089     UK02-CGBA020293-L
314939    UK02-SGBA006251-L
Name: account_number, dtype: str

In [10]:
df_bob['account_number'] = df_bob['account_number'].str.split('-').str[1]

### Verify Account Number Transformation
Sample the transformed account numbers to confirm the extraction worked correctly.

In [11]:
df_bob['account_number'].sample(5)

41068     CGBA003652
126069    SGBA005041
148810    SGBA012048
144132    SGBA019758
36195     CGBA000004
Name: account_number, dtype: object

### Remove Records with Missing Keys
Filter out rows where account_number or branch are null, as these are critical identifiers.

In [12]:
df_bob = df_bob[(df_bob['account_number'].notna()) & (df_bob['branch'].notna())]

### Data Quality Check - Post Filter
Verify the dataset shape after filtering and identify any remaining missing values.

In [13]:
print(df_bob.shape)
df_bob.isna().sum()[df_bob.isna().sum() > 0]

(198977, 26)


company_sizing            1279
postal_code                  2
vat_number                 214
bpg                       2408
msdyn_product_number      1255
product_name              1255
service_interval         57126
unit_amount               2394
billing_interval         12016
billing_period           12016
machine                 112839
machine_variant         112835
chemistry               132429
dtype: int64

In [14]:
df_bob.columns

Index(['account_number', 'company_sizing', 'postal_code', 'branch',
       'vat_number', 'agreement_number', 'agreement_start_date',
       'agreement_end_date', 'renewal_type', 'agreement_type',
       'line_of_business', 'system_status', 'product_bob', 'fee_bob',
       'total_bob', 'is_bob', 'bpg', 'msdyn_product_number', 'product_name',
       'service_interval', 'unit_amount', 'billing_interval', 'billing_period',
       'machine', 'machine_variant', 'chemistry'],
      dtype='str')

In [15]:
df_bob[['account_number', 'branch', 'agreement_number', 'line_of_business']].sample(5)

,account_number,branch,agreement_number,line_of_business
96742,CGBA024843,Derby,GBC2555721-Machine Services-81033,Machine Services
52517,CGBA016879,Bristol,GBC2550671-Chemistry,Chemistry
147821,SGBA007892,Chester,GBC2501021-Machine Services-13414,Machine Services
110230,CGBA020595,West London,GBC2547140-Machine Services-328483,Machine Services
225327,SGBA221227,West London,GBC4021222-Auto waste,Auto waste


### Investigate vat_number Field
Check for null values and 'Unknown' entries in the vat_number column to decide if it should be dropped.

In [16]:
print(df_bob['vat_number'].isna().sum())

df_bob[df_bob['vat_number'] == 'Unknown']['vat_number'].count()

214


np.int64(167027)

### Drop Non-Essential Columns
Remove vat_number and postal_code columns as they have insufficient data quality for analysis.

In [17]:
df_bob.drop(columns=['vat_number', 'postal_code'], inplace=True)

### Remove Duplicates (Post-Cleanup)
Run deduplication again after removing columns and transforming data, as new duplicates may have been created.

In [18]:
print(f'Duplicate rows: {df_bob.duplicated().sum()}')

df_bob.drop_duplicates(inplace=True)

print(f'Duplicate rows after dropping: {df_bob.duplicated().sum()}')

Duplicate rows: 2221
Duplicate rows after dropping: 0


### Calculate Agreement Duration
Create a new column calculating the duration of each agreement by subtracting start date from end date.

In [19]:
df_bob['agreement_duration'] = df_bob['agreement_end_date'] - df_bob['agreement_start_date']

In [20]:
df_bob[['agreement_start_date', 'agreement_end_date', 'agreement_duration']].sample(5)

,agreement_start_date,agreement_end_date,agreement_duration
40495,2021-01-13,2026-01-12,1825 days
125361,2025-02-28,2027-07-30,882 days
200936,2022-07-29,2026-07-28,1460 days
148553,2018-03-11,2026-03-10,2921 days
7519,2023-02-20,2026-02-19,1095 days


### Data Quality Check - Final
Display final shape and any remaining missing values before export.


In [21]:
print(df_bob.shape)
df_bob.isna().sum()[df_bob.isna().sum() > 0]

(196756, 25)


company_sizing            1239
bpg                       2399
msdyn_product_number      1246
product_name              1246
service_interval         55349
unit_amount               2386
billing_interval         12003
billing_period           12003
machine                 110812
machine_variant         110808
chemistry               130372
dtype: int64

In [22]:
df_bob.info()

<class 'pandas.DataFrame'>
Index: 196756 entries, 0 to 376788
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype          
---  ------                --------------   -----          
 0   account_number        196756 non-null  object         
 1   company_sizing        195517 non-null  str            
 2   branch                196756 non-null  str            
 3   agreement_number      196756 non-null  str            
 4   agreement_start_date  196756 non-null  datetime64[us] 
 5   agreement_end_date    196756 non-null  datetime64[us] 
 6   renewal_type          196756 non-null  str            
 7   agreement_type        196756 non-null  str            
 8   line_of_business      196756 non-null  str            
 9   system_status         196756 non-null  str            
 10  product_bob           196756 non-null  float64        
 11  fee_bob               196756 non-null  float64        
 12  total_bob             196756 non-null  float64        
 13  

### Analyze Categorical Variables
Value counts for key categorical fields:
- system_status: Status of the billing system
- is_bob: Whether this is a BoB agreement
- billing_period: Frequency of billing cycles
- chemistry: Type of product chemistry

In [23]:
df_bob['system_status'].value_counts()

system_status
Active      195648
Estimate      1056
Canceled        51
Expired          1
Name: count, dtype: int64

In [24]:
df_bob['is_bob'].value_counts()

is_bob
Yes    196566
ST        190
Name: count, dtype: int64

In [25]:
df_bob['billing_period'].value_counts()

billing_period
monthly    184748
yearly          5
Name: count, dtype: int64

In [26]:
df_bob['chemistry'].value_counts()

chemistry
D60                     23417
KLEEN1000M               8114
KLEEN7350J               7546
KLEEN1810J               5072
KLEEN7760M               4046
KLEEN3460S               3249
KLEEN350:L               2452
KLEEN1200J               1931
KLEEN7960S               1743
KLEEN4760N               1456
KLEEN1020N               1035
SOLV757                   983
KLEEN6810S                830
KLEEN7540N                725
Natural Solvent           717
Resinpro                  647
KLEEN9510L                611
KLEEN3300N                543
PCT Water                 258
KLEEN7840S                219
KLEEN2020N                151
KLEEN1100V                101
KLEEN6810W                 81
KLEEN3090N                 77
Butyl Diglycol             74
KLEEN1640K                 72
KLEEN5160S                 65
Samsol                     48
Safety Solvent Clean       23
KLEEN350:N                 22
KLEEN5120S                 16
KLEEN4150S                 14
KLEEN4760S                  9


### Reorganize Columns
Reorder columns into a logical grouping: identifiers → agreement details → product info → machine specs.

In [27]:
bob_column_order = [
    "account_number",
    "company_sizing",
    "branch",

    "agreement_number",
    "agreement_start_date",
    "agreement_end_date",
    "agreement_duration",

    "agreement_type",
    "renewal_type",
    "line_of_business",
    "system_status",

    "product_bob",
    "fee_bob",
    "total_bob",
    "is_bob",

    "bpg",
    "msdyn_product_number",
    "product_name",
    "service_interval",

    "unit_amount",
    "billing_interval",
    "billing_period",

    "machine",
    "machine_variant",
    "chemistry"
]
df_bob = df_bob[bob_column_order]

In [28]:
df_bob.duplicated().sum(), df_bob.shape

(np.int64(0), (196756, 25))

### Save Cleaned BoB Data
Export the cleaned BoB dataset to Excel for use in downstream analysis.

In [29]:
df_bob.to_excel(clean_bob_file, index=False)
print(f'Excel file saved to this path: {clean_bob_file}')

Excel file saved to this path: dataset_2/bob_clean.xlsx


## Retention Data Processing

### Load Retention Data
Read the CSV file with latin1 encoding (required for special characters in the data).

In [30]:
df_retention = pd.read_csv(raw_retention_file, encoding='latin1')

### Data Quality Check - Initial
Display dataset structure and data types.

In [31]:
df_retention.info()

<class 'pandas.DataFrame'>
RangeIndex: 85411 entries, 0 to 85410
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Case ID                    85411 non-null  str    
 1   Case Title                 85411 non-null  str    
 2   Country                    85411 non-null  str    
 3   Pull VAN                   85411 non-null  float64
 4   New VAN                    85411 non-null  float64
 5   VAN                        84980 non-null  float64
 6   Number of Contracts        71249 non-null  float64
 7   Machines                   71249 non-null  float64
 8   Branch                     77280 non-null  str    
 9   Customer Account Number    77949 non-null  str    
 10  Customer Name              77949 non-null  str    
 11  Customer Since             0 non-null      float64
 12  Agreement End Date         85205 non-null  str    
 13  Pull Type                  50204 non-null  str    
 14  C

### Standardize Column Names
Convert column names to lowercase and replace spaces with underscores for consistent naming conventions.

In [32]:
df_retention.columns = df_retention.columns.str.strip().str.lower().str.replace(' ', '_')
df_retention.columns

Index(['case_id', 'case_title', 'country', 'pull_van', 'new_van', 'van',
       'number_of_contracts', 'machines', 'branch', 'customer_account_number',
       'customer_name', 'customer_since', 'agreement_end_date', 'pull_type',
       'case_type', 'risk', 'current_status', 'resolution_status',
       'number_of_repair_cases', 'number_of_overdueservices', 'companysize',
       'customer_tier', 'case_origin', 'case_creation_date', 'resolved_time',
       'registered_time', 'resolved_date', 'registered_date',
       'expected_pull_date'],
      dtype='str')

### Remove Duplicate Rows
Identify and remove duplicate records from the retention dataset.

In [33]:
print(f'Duplicate rows: {df_retention.duplicated().sum()}')

df_retention.drop_duplicates(inplace=True)

print(f'Duplicate rows after dropping: {df_retention.duplicated().sum()}')

Duplicate rows: 2944
Duplicate rows after dropping: 0


### Clean String Columns
Strip whitespace from all string and object-type columns.

In [34]:
strip_all_string_cols(df_retention)

### Drop Non-Essential Columns
Remove columns with insufficient data:
- customer_since: Contains nothing
- country: Irrelevant as all are UK
- number_of_overdueservices: Only missing values and 0s.

In [35]:
df_retention.drop(columns=['customer_since', 'country', 'number_of_overdueservices'], inplace=True)

### Rename Columns for Clarity
Standardize column names:
- customer_account_number → account_number (for merge compatibility)
- companysize → company_size
- machines → number_of_machines

Remove records with missing account numbers since we need this for merging.

In [36]:
df_retention.rename(columns={
    'customer_account_number': 'account_number',
    'companysize': 'company_size',
    'machines': 'number_of_machines',
    }, 
    inplace=True)

df_retention = df_retention[df_retention['account_number'].notna()]

### Data Quality Check - Nulls
Identify columns with missing values.

In [37]:
df_retention.isna().sum()[df_retention.isna().sum() > 0]

van                         117
number_of_contracts       13906
number_of_machines        13906
branch                     1693
agreement_end_date          123
pull_type                 34348
risk                      36374
number_of_repair_cases    14433
company_size              16972
customer_tier             22657
case_origin               20708
resolved_time              8205
resolved_date              8205
expected_pull_date          169
dtype: int64

In [38]:
df_retention.info()

<class 'pandas.DataFrame'>
Index: 75511 entries, 3000 to 85410
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   case_id                 75511 non-null  str    
 1   case_title              75511 non-null  str    
 2   pull_van                75511 non-null  float64
 3   new_van                 75511 non-null  float64
 4   van                     75394 non-null  float64
 5   number_of_contracts     61605 non-null  float64
 6   number_of_machines      61605 non-null  float64
 7   branch                  73818 non-null  str    
 8   account_number          75511 non-null  str    
 9   customer_name           75511 non-null  str    
 10  agreement_end_date      75388 non-null  str    
 11  pull_type               41163 non-null  str    
 12  case_type               75511 non-null  str    
 13  risk                    39137 non-null  str    
 14  current_status          75511 non-null  str    
 15

### Check for Duplicates
Count remaining duplicate rows and create a copy for further processing.

In [39]:
print(df_retention.duplicated().sum())

df = df_retention.copy()

0


### Identify Date Columns
List all date and datetime columns to process them systematically.

In [40]:
dates = [
    'agreement_end_date', 'case_creation_date', 'resolved_time', 
    'registered_time', 'resolved_date', 'registered_date', 'expected_pull_date'
        ]

df[dates].sample(10)

,agreement_end_date,case_creation_date,resolved_time,registered_time,resolved_date,registered_date,expected_pull_date
81893,23-11-2022 00:00,21-10-2022 08:05,2:00:00,10:05:16,23-11-2022,21-10-2022,23-Nov-22
67808,30-09-2026 00:00,13-08-2024 07:39,10:47:09,9:39:22,28-08-2024,13-08-2024,16-Aug-24
44323,1/4/2026 0:00,24-03-2021 12:48,5:16:06,14:48:20,1/8/2021,24-03-2021,12-Aug-21
44043,24-08-2023 00:00,9/6/2023 10:53,10:35:23,12:53:36,17-08-2023,9/6/2023,11-Dec-23
6034,18-08-2026 23:00,8/5/2025 13:21,NaN,15:21:55,NaN,8/5/2025,19-May-26
82219,5/3/2025 0:00,15-08-2024 10:16,2:00:00,12:16:38,5/3/2025,15-08-2024,5-Mar-25
82227,13-07-2023 00:00,6/4/2023 9:04,2:00:00,11:04:40,13-07-2023,6/4/2023,13-Jul-23
21865,16-12-2025 00:00,30-07-2025 10:59,2:00:00,13:59:21,12/8/2025,30-07-2025,12-Aug-25
82705,9/6/2023 0:00,23-01-2023 14:15,2:00:00,16:15:56,9/6/2023,23-01-2023,9-Jun-23
68350,17-04-2026 00:00,8/3/2024 8:22,12:26:51,10:22:06,16-04-2024,8/3/2024,16-Apr-24


### Check Date Quality
Count missing values in each date column.

In [41]:
df[dates].isna().sum()

agreement_end_date     123
case_creation_date       0
resolved_time         8205
registered_time          0
resolved_date         8205
registered_date          0
expected_pull_date     169
dtype: int64

### Parse & Combine Datetime Columns
Convert date strings using dayfirst=True (European format):
- agreement_end_date: Direct conversion
- case_creation_date: Direct conversion
- registered_date: Combine with registered_time
- resolved_date: Combine with resolved_time
- expected_pull_date: Special format '%d-%b-%y'

Drop the separate time columns after combining.

In [42]:
df['agreement_end_date'] = pd.to_datetime(df['agreement_end_date'], dayfirst=True, errors='coerce')
df['case_creation_date'] = pd.to_datetime(df['case_creation_date'], dayfirst=True, errors='coerce')

df['registered_date'] = pd.to_datetime(
    df['registered_date'].astype(str) + " " + df['registered_time'].astype(str),
    dayfirst=True,
    errors='coerce'
)

df['resolved_date'] = pd.to_datetime(
    df['resolved_date'].astype(str) + " " + df['resolved_time'].astype(str),
    dayfirst=True,
    errors='coerce'
)

df['expected_pull_date'] = pd.to_datetime(df['expected_pull_date'], format='%d-%b-%y', errors='coerce')

df.drop(columns=['resolved_time', 'registered_time'], inplace=True)

### Recheck Date Quality
Verify all date conversions completed successfully.


In [43]:
dates.remove('registered_time')
dates.remove('resolved_time')

df[dates].isna().sum()

agreement_end_date    28666
case_creation_date    28497
resolved_date         41043
registered_date       28595
expected_pull_date      169
dtype: int64

In [44]:
df[dates].sample(10)

,agreement_end_date,case_creation_date,resolved_date,registered_date,expected_pull_date
74445,NaT,2019-08-13 07:12:00,2019-11-07 02:00:00,2019-08-13 09:12:41,2019-11-07
5168,2026-09-21 23:00:00,2025-06-25 13:39:00,NaT,2025-06-25 15:39:13,2026-06-22
76216,2020-09-15 15:24:00,NaT,2020-08-01 02:00:00,NaT,2020-08-01
34198,2025-09-29 00:00:00,NaT,2021-08-01 05:02:16,NaT,2020-09-30
58593,NaT,2024-12-13 08:23:00,2025-02-04 11:44:22,2024-12-13 10:23:38,2025-02-04
71135,2020-07-22 15:27:00,2018-12-13 12:28:00,NaT,2018-12-13 14:28:03,2019-01-22
70263,2019-08-14 15:22:00,NaT,2019-08-09 02:00:00,NaT,2019-08-09
63149,NaT,2022-05-26 10:12:00,2022-06-10 02:00:00,2022-05-26 12:12:37,2022-06-10
65619,NaT,2024-08-29 11:53:00,2024-09-02 02:00:00,2024-08-29 13:53:59,2024-09-02
22559,2025-10-21 23:00:00,2024-05-16 08:39:00,NaT,2025-05-16 10:00:00,2025-07-28


### Convert Integer Columns
Convert count columns to Int8 (nullable integer type):
- number_of_contracts
- number_of_machines
- number_of_repair_cases

In [45]:
df['number_of_contracts'] = df['number_of_contracts'].astype(dtype='Int8')
df['number_of_machines'] = df['number_of_machines'].astype(dtype='Int8')
df['number_of_repair_cases'] = df['number_of_repair_cases'].astype(dtype='Int8')

### Data Quality Check - After Type Conversion
Display final data types and structure.

In [46]:
df.info()

<class 'pandas.DataFrame'>
Index: 75511 entries, 3000 to 85410
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   case_id                 75511 non-null  str           
 1   case_title              75511 non-null  str           
 2   pull_van                75511 non-null  float64       
 3   new_van                 75511 non-null  float64       
 4   van                     75394 non-null  float64       
 5   number_of_contracts     61605 non-null  Int8          
 6   number_of_machines      61605 non-null  Int8          
 7   branch                  73818 non-null  str           
 8   account_number          75511 non-null  str           
 9   customer_name           75511 non-null  str           
 10  agreement_end_date      46845 non-null  datetime64[us]
 11  pull_type               41163 non-null  str           
 12  case_type               75511 non-null  str           
 13 

### Analyze Customer Tier Values
Count unique values in customer_tier column to identify inconsistencies.

In [47]:
df['customer_tier'].value_counts()

customer_tier
Platinum       28835
Diamond        10832
Key Account     9408
Platinum +      2703
Platinum+       1076
Name: count, dtype: int64

### Standardize Customer Tier
Replace variations of "Platinum +" with standardized "Platinum Plus" naming.

In [48]:
df['customer_tier'] = df['customer_tier'].replace(
    {"Platinum +": "Platinum Plus", "Platinum+": "Platinum Plus"}
)

df['customer_tier'].value_counts()


customer_tier
Platinum         28835
Diamond          10832
Key Account       9408
Platinum Plus     3779
Name: count, dtype: int64

### Check for Null Tiers
Verify no missing customer_tier values after standardization.

In [49]:
df['customer_tier'].isna().sum()

np.int64(22657)

### Extract Account IDs from Retention Data
Similar to BoB data, split account numbers to extract ID portion after hyphen.

In [50]:
df['account_number'].sample(5)

81335    UK02-CGBA031769-L
34274    UK02-CGBA028357-L
56663    UK02-CGBA031211-L
5988     UK02-CGBA017214-L
8099     UK02-SGBA106468-L
Name: account_number, dtype: str

In [51]:
df['account_number'] = df['account_number'].str.split('-').str[1]

In [52]:
df['account_number'].sample(5)

64619    CGBA206261
14041    CGBA205714
70651    CGBA111883
5911     CGBA017192
78733    CGBA012665
Name: account_number, dtype: object

### Preview Full Retention Dataset
Sample 5 complete rows to review all transformations.

In [53]:
df.sample(5)

,case_id,case_title,pull_van,new_van,van,number_of_contracts,number_of_machines,branch,account_number,customer_name,...,current_status,resolution_status,number_of_repair_cases,company_size,customer_tier,case_origin,case_creation_date,resolved_date,registered_date,expected_pull_date
28247,CAS-19495-J6M4R7,PULL: FULL ACCOUNT M100 & 158,0.0,0.0,0.0,<NA>,<NA>,Bristol,CGBA014709,Acklea- A Division of SHB Hire Limited,...,Formal Notification,Customer Saved,0,NaN,Platinum,Site Visit,2018-04-27 11:35:00,2018-05-02 12:02:17,2018-04-27 13:35:57,2018-04-27
7964,CAS-85390-B0W1T1,3.3 Service issue,0.0,0.0,423.0,1,1,Montrose,CGBA119088,SWR - Arnold Clark,...,Waiting for Details,OPEN - In Progress,<NA>,>1000,NaN,Email,NaT,NaT,NaT,2026-05-16
10787,CAS-31359-S2J2H9,Service Claim,0.0,0.0,748.8,1,1,Stirling,SGBA111381,Arnold Clark Linwood Hyundai,...,In Progress,OPEN - In Progress,<NA>,>1000,Key Account,Internal Email,2025-09-25 08:23:00,NaT,2025-07-23 10:00:00,2026-04-28
29784,CAS-89259-C6J9Q6,Risk - Debt,0.0,0.0,0.0,<NA>,<NA>,West Bromwich,CGBA117334,Lynx Motors Engineering Limited,...,Investigation,Converted to Cancellation,0,NaN,NaN,Customer Service Team Leader,2021-03-18 11:21:00,2021-08-01 05:23:23,2021-03-18 13:21:57,2023-07-28
62527,CAS-82362-Z4G4Y7,Waste has moved to PATOS- Machine not needed,663.4,0.0,663.4,<NA>,<NA>,Exeter,CGBA105497,Morris Leslie Plant Hire,...,Pull In Progress,Customer Lost,0,100-249,NaN,Account Manager,2020-09-21 13:51:00,2020-10-12 02:00:00,2020-09-21 15:51:26,2020-10-12


### Final Null Check
Identify any remaining missing values before export.

In [54]:
df.isna().sum()[df.isna().sum() > 0]

van                         117
number_of_contracts       13906
number_of_machines        13906
branch                     1693
agreement_end_date        28666
pull_type                 34348
risk                      36374
number_of_repair_cases    14433
company_size              16972
customer_tier             22657
case_origin               20708
case_creation_date        28497
resolved_date             41043
registered_date           28595
expected_pull_date          169
dtype: int64

### Save Cleaned Retention Data
Export the cleaned retention dataset to Excel for downstream analysis.

In [55]:
df.to_excel(clean_retention_file, index=False)
print(f'Excel file saved to this path: {clean_retention_file}')

Excel file saved to this path: dataset_2/retention_clean.xlsx


## Data Merging

### Reload Cleaned Datasets
Import both cleaned datasets for merging process.

In [120]:
df_bob = pd.read_excel(clean_bob_file)
df_retention = pd.read_excel(clean_retention_file)

### Review BoB Structure
Display final BoB dataset structure, data types, and row count.

In [121]:
df_bob.info()

<class 'pandas.DataFrame'>
RangeIndex: 196756 entries, 0 to 196755
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   account_number        196756 non-null  str           
 1   company_sizing        195517 non-null  str           
 2   branch                196756 non-null  str           
 3   agreement_number      196756 non-null  str           
 4   agreement_start_date  196756 non-null  datetime64[us]
 5   agreement_end_date    196756 non-null  datetime64[us]
 6   agreement_duration    196756 non-null  int64         
 7   agreement_type        196756 non-null  str           
 8   renewal_type          196756 non-null  str           
 9   line_of_business      196756 non-null  str           
 10  system_status         196756 non-null  str           
 11  product_bob           196756 non-null  float64       
 12  fee_bob               196756 non-null  float64       
 13  total_bob 

### Review Retention Structure
Display final retention dataset structure, data types, and row count.

In [122]:
df_retention.info()

<class 'pandas.DataFrame'>
RangeIndex: 75511 entries, 0 to 75510
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   case_id                 75511 non-null  str           
 1   case_title              75504 non-null  str           
 2   pull_van                75511 non-null  float64       
 3   new_van                 75511 non-null  float64       
 4   van                     75394 non-null  float64       
 5   number_of_contracts     61605 non-null  float64       
 6   number_of_machines      61605 non-null  float64       
 7   branch                  73818 non-null  str           
 8   account_number          75511 non-null  str           
 9   customer_name           75511 non-null  str           
 10  agreement_end_date      46845 non-null  datetime64[us]
 11  pull_type               41163 non-null  str           
 12  case_type               75511 non-null  str           
 1

### Clean the 'line_of_business' column for pivoting
Converts "Machine Services" to "machine_services"

In [123]:
df_bob['line_of_business'] = df_bob['line_of_business'].str.lower().str.replace(' ', '_')

### Overall Account Aggregation (Using our finalized dict)

In [124]:
account_agg_dict = {
    'agreement_number': 'count', 
    'agreement_start_date': 'min', 
    'agreement_end_date': 'max', 
    'company_sizing': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'branch': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'agreement_type': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'renewal_type': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'system_status': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'product_bob': 'sum',
    'fee_bob': 'sum',
    'total_bob': 'sum',
    'product_name': 'nunique',  
    'machine': 'nunique',  
    'machine_variant': 'nunique',
    'chemistry': 'nunique'
}

bob_account_level = df_bob.groupby('account_number').agg(account_agg_dict).reset_index()

In [125]:
bob_account_level.rename(columns={
    'agreement_number': 'total_agreements',
    'agreement_start_date': 'first_agreement_start',
    'agreement_end_date': 'last_agreement_end',
    'product_name': 'unique_products_count',
    'machine': 'unique_machines_count',
    'machine_variant': 'unique_machine_variants_count',
    'chemistry': 'unique_chemistries_count'
}, inplace=True)

In [126]:
branch_counts = df_bob.groupby('account_number')['branch'].nunique().reset_index(name='unique_branches_count')
bob_account_level = pd.merge(bob_account_level, branch_counts, on='account_number')

### Pivot the LOB-Specific Financials & Counts

In [127]:
bob_lob_pivot = df_bob.pivot_table(
    index='account_number',
    columns='line_of_business',
    values=['total_bob', 'agreement_number', 'unit_amount', 'service_interval'], 
    aggfunc={
        'total_bob': 'sum',          
        'agreement_number': 'count',  
        'unit_amount': 'mean',    
        'service_interval': 'mean'  
    },
    fill_value=0 
)

bob_lob_pivot.rename(columns={
    'agreement_number': 'agreement_count'
}, inplace=True)

Flatten the column names (e.g., this turns a tuple into 'machine_services_total_bob')

In [128]:
bob_lob_pivot.columns = [f"{col[1]}_{col[0]}" for col in bob_lob_pivot.columns]
bob_lob_pivot.reset_index(inplace=True)

### Merge into one final, flat BoB table

In [129]:
bob_final = pd.merge(bob_account_level, bob_lob_pivot, on='account_number', how='inner')

In [130]:
bob_final.shape

(23771, 41)

In [131]:
bob_final.sample(10)

,account_number,total_agreements,first_agreement_start,last_agreement_end,company_sizing,branch,agreement_type,renewal_type,system_status,product_bob,fee_bob,total_bob,unique_products_count,unique_machines_count,unique_machine_variants_count,unique_chemistries_count,unique_branches_count,allied_agreement_count,auto_waste_agreement_count,chemistry_agreement_count,kleenwaste_agreement_count,machine_services_agreement_count,oil_agreement_count,allied_service_interval,auto_waste_service_interval,chemistry_service_interval,kleenwaste_service_interval,machine_services_service_interval,oil_service_interval,allied_total_bob,auto_waste_total_bob,chemistry_total_bob,kleenwaste_total_bob,machine_services_total_bob,oil_total_bob,allied_unit_amount,auto_waste_unit_amount,chemistry_unit_amount,kleenwaste_unit_amount,machine_services_unit_amount,oil_unit_amount
17764,SGBA026021,3,2021-05-20,2026-05-19,>1000,Dinnington,Scheduled Billing,Automatic Renewal,Active,10945.120,1000.080,11945.200,1,1,1,1,1,0,0,0,0,3,0,0.000,0.000,0.000,0.000,7.000,0.000,0.000,0.000,0.000,0.000,11945.200,0.000,0.000,0.000,0.000,0.000,304.033,0.000
21975,SGBA205425,6,2022-03-31,2026-03-30,100-249,Bristol,Scheduled Billing,Automatic Renewal,Active,12164.520,750.000,12914.520,2,2,1,1,1,0,0,0,0,6,0,0.000,0.000,0.000,0.000,8.000,0.000,0.000,0.000,0.000,0.000,12914.520,0.000,0.000,0.000,0.000,0.000,168.952,0.000
11203,SGBA003680,6,2025-08-06,2027-08-29,20-49,Cardiff,Scheduled Billing,Automatic Renewal,Active,43707.380,2250.000,45957.380,2,2,1,1,1,0,0,0,0,6,0,0.000,0.000,0.000,0.000,3.000,0.000,0.000,0.000,0.000,0.000,45957.380,0.000,0.000,0.000,0.000,0.000,607.047,0.000
21507,SGBA123870,3,2023-03-30,2026-03-29,1-9,Wigan,Scheduled Billing,Automatic Renewal,Active,4268.640,750.000,5018.640,1,1,1,0,1,0,0,0,0,3,0,0.000,0.000,0.000,0.000,8.000,0.000,0.000,0.000,0.000,0.000,5018.640,0.000,0.000,0.000,0.000,0.000,118.573,0.000
23696,SGBA221762,6,2025-04-30,2028-05-29,1-9,Cardiff,Scheduled Billing,Automatic Renewal,Active,16970.800,656.280,17627.080,2,2,1,2,1,0,0,0,0,6,0,0.000,0.000,0.000,0.000,19.500,0.000,0.000,0.000,0.000,0.000,17627.080,0.000,0.000,0.000,0.000,0.000,235.707,0.000
334,000253474,6,2026-01-19,2029-01-19,Unknown,Bedford,Scheduled Billing,Automatic Renewal,Active,175.440,0.000,175.440,5,0,0,0,1,0,6,0,0,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,175.440,0.000,0.000,0.000,0.000,0.000,2.437,0.000,0.000,0.000,0.000
5640,CGBA026779,5,2019-05-22,2028-07-31,10-19,Dinnington,Scheduled Billing,Automatic Renewal,Active,7548.080,2080.080,9628.160,4,1,1,1,1,0,2,0,0,3,0,0.000,9.000,0.000,0.000,2.000,0.000,0.000,1800.000,0.000,0.000,7828.160,0.000,0.000,75.000,0.000,0.000,217.450,0.000
23457,SGBA220141,2,2024-11-18,2027-11-17,1-9,Exeter,Scheduled Billing,Automatic Renewal,Active,3954.900,0.000,3954.900,2,1,1,1,1,0,0,1,0,1,0,0.000,0.000,8.000,0.000,8.000,0.000,0.000,0.000,0.000,0.000,3954.900,0.000,0.000,0.000,0.000,0.000,0.000,0.000
21985,SGBA205676,17,2023-10-01,2026-09-30,>1000,Derby,Scheduled Billing,Automatic Renewal,Active,16016.360,1493.280,17509.640,6,2,1,1,1,0,9,0,0,6,2,0.000,10.000,0.000,0.000,8.500,26.000,0.000,6051.120,0.000,0.000,11016.920,441.600,0.000,56.029,0.000,0.000,132.273,18.400
14512,SGBA014814,7,2017-09-02,2028-06-15,100-249,Bristol,PATOS,Automatic Renewal,Active,10087.320,986.280,11073.600,2,2,1,2,1,0,0,0,0,7,0,0.000,0.000,0.000,0.000,18.286,0.000,0.000,0.000,0.000,0.000,11073.600,0.000,0.000,0.000,0.000,0.000,264.253,0.000


In [132]:
bob_final[bob_final['account_number'] == 'CGBA017192']

,account_number,total_agreements,first_agreement_start,last_agreement_end,company_sizing,branch,agreement_type,renewal_type,system_status,product_bob,fee_bob,total_bob,unique_products_count,unique_machines_count,unique_machine_variants_count,unique_chemistries_count,unique_branches_count,allied_agreement_count,auto_waste_agreement_count,chemistry_agreement_count,kleenwaste_agreement_count,machine_services_agreement_count,oil_agreement_count,allied_service_interval,auto_waste_service_interval,chemistry_service_interval,kleenwaste_service_interval,machine_services_service_interval,oil_service_interval,allied_total_bob,auto_waste_total_bob,chemistry_total_bob,kleenwaste_total_bob,machine_services_total_bob,oil_total_bob,allied_unit_amount,auto_waste_unit_amount,chemistry_unit_amount,kleenwaste_unit_amount,machine_services_unit_amount,oil_unit_amount
3718,CGBA017192,62,2017-10-04,2028-07-30,50-99,Maidstone,Scheduled Billing,Automatic Renewal,Active,59102.200,5504.160,64606.360,17,6,2,2,2,3,24,3,0,30,2,4.000,13.125,4.000,0.000,5.600,0.000,2046.960,7237.800,580.320,0.000,54741.280,0.000,56.860,25.131,16.120,0.000,122.388,0.000


In [133]:
bob_final.isna().sum()[bob_final.isna().sum() > 0]

Series([], dtype: int64)

### Retention Table working

In [134]:
df_retention.shape

(75511, 24)

In [135]:
df_retention.isna().sum()[df_retention.isna().sum() > 0]

case_title                    7
van                         117
number_of_contracts       13906
number_of_machines        13906
branch                     1693
agreement_end_date        28666
pull_type                 34348
risk                      36374
number_of_repair_cases    14433
company_size              16972
customer_tier             22657
case_origin               20708
case_creation_date        28497
resolved_date             41043
registered_date           28595
expected_pull_date          169
dtype: int64

In [136]:
df_retention['resolution_status'].value_counts()

resolution_status
Customer Saved               34995
Customer Lost                23161
Converted to Cancellation     9142
OPEN - In Progress            8158
OPEN - Pull Confirmed           47
Unknown                          8
Name: count, dtype: int64

In [137]:
status_mapping = {
    'Converted to Cancellation': 'Customer Lost',
    'OPEN - In Progress': 'OPEN',
    'OPEN - Pull Confirmed': 'OPEN'
}

df_retention['resolution_status'] = df_retention['resolution_status'].replace(status_mapping)

In [138]:
df_retention['resolution_status'].value_counts()

resolution_status
Customer Saved    34995
Customer Lost     32303
OPEN               8205
Unknown               8
Name: count, dtype: int64

In [139]:
df_retention['current_status'].value_counts()

current_status
Formal Notification      33772
Pull Complete            13468
Pull In Progress          6195
In Progress               6117
Resolved                  4886
Problem Solved            4401
Pull Completed            1808
Waiting for Details       1766
Pull Quote Created        1647
Investigation             1032
Pull Confirmed             360
Negotiation                 20
Information Provided        19
On Hold                     14
New Agreement Created        6
Name: count, dtype: int64

In [140]:
df_retention['is_churn_case'] = (df_retention['resolution_status'] == 'Customer Lost').astype(int)

In [142]:
retention_agg_dict = {
    'case_id': 'nunique',
    'pull_van': 'sum',
    'new_van': 'sum',
    'van': 'sum',
    'number_of_repair_cases': 'sum',
    'customer_tier': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'resolution_status': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'pull_type': lambda x: x.mode()[0] if not x.mode().empty else None,
    'case_type': lambda x: x.mode()[0] if not x.mode().empty else None,
    'risk': lambda x: x.mode()[0] if not x.mode().empty else None,
    'case_origin': lambda x: x.mode()[0] if not x.mode().empty else None,
    'case_creation_date': ['min', 'max'],
    'registered_date': ['min', 'max'],
    'expected_pull_date': 'max',
    'is_churn_case': 'max',
    'resolved_date': 'max'
}

retention_final = df_retention.groupby('account_number').agg(retention_agg_dict).reset_index()

In [143]:
retention_final.columns = [f'{col[0]}_{col[1]}' if col[1] else col[0] for col in retention_final.columns]

In [144]:
retention_final.rename(columns={
    'case_id_nunique': 'total_cases',
    'number_of_repair_cases_sum': 'total_repair_cases',
    'pull_van_sum': 'total_pull_van',
    'new_van_sum': 'total_new_van',
    'van_sum': 'total_van',
    'registered_date_min': 'first_registered_date',
    'registered_date_max': 'last_registered_date',
    'case_creation_date_min': 'first_case_creation_date',
    'case_creation_date_max': 'last_case_creation_date',
    'expected_pull_date_max': 'latest_expected_pull_date',
    'customer_tier_<lambda>': 'customer_tier',
    'resolved_date_max': 'latest_resolved_date',
    'case_type_<lambda>': 'case_type',
    'pull_type_<lambda>': 'pull_type',
    'risk_<lambda>': 'risk_level',
    'case_origin_<lambda>': 'case_origin',
    'is_churn_case_max': 'churn_label',
    'resolution_status_<lambda>': 'resolution_status'
}, inplace=True)

In [150]:
retention_final.shape, bob_final.shape

((21526, 19), (23771, 41))

In [146]:
retention_final.sample(5)

,account_number,total_cases,total_pull_van,total_new_van,total_van,total_repair_cases,customer_tier,resolution_status,pull_type,case_type,risk_level,case_origin,first_case_creation_date,last_case_creation_date,first_registered_date,last_registered_date,latest_expected_pull_date,churn_label,latest_resolved_date
18392,CGBA118511,1,0.000,3240.607,3240.607,0.000,Platinum,Customer Saved,NaN,Risk,Site Access,NaN,2023-06-27 06:53:00,2023-06-27 06:53:00,2023-06-27 08:53:52,2023-06-27 08:53:52,2023-09-29,0,2023-07-11 13:49:45
15075,CGBA032559,1,1724.316,0.000,1724.316,0.000,Platinum,Customer Lost,Full,Cancellation,NaN,BSU/ERP customer services,NaT,NaT,NaT,NaT,2021-12-02,1,2021-12-02 02:00:00
8914,CGBA019170,1,15712.799,0.000,15712.799,0.000,Platinum,Customer Lost,Full,Cancellation,NaN,Notice in Writing,2023-11-16 13:02:00,2023-11-16 13:02:00,2023-11-16 15:02:54,2023-11-16 15:02:54,2024-05-16,1,NaT
3617,CGBA008180,2,1307.477,264.000,1571.477,0.000,Unknown,Customer Lost,Full,Cancellation,Contract Enquiry,Account Manager,2022-11-16 15:56:00,2022-11-16 15:56:00,2022-11-16 17:56:06,2022-11-16 17:56:06,2022-11-16,1,2021-08-01 05:24:07
14460,CGBA031520,2,0.000,624.000,624.000,0.000,Key Account,Customer Saved,NaN,Cancellation,Contract Expiring Soon,Site Visit,2019-02-25 09:55:00,2019-02-25 09:55:00,2019-02-25 11:55:50,2019-02-25 11:55:50,2019-07-20,0,2021-10-02 11:57:27


### Actual Merging the 2 tables

In [169]:
master_df = pd.merge(bob_final, retention_final, on='account_number', how='inner')

In [170]:
master_df.shape

(5628, 59)

In [171]:
master_df.isna().sum()[master_df.isna().sum() > 0]

pull_type                    1835
risk_level                   2109
case_origin                  1179
first_case_creation_date     1374
last_case_creation_date      1374
first_registered_date        1390
last_registered_date         1390
latest_expected_pull_date      36
latest_resolved_date         2449
dtype: int64

In [172]:
master_df.sample(5)

,account_number,total_agreements,first_agreement_start,last_agreement_end,company_sizing,branch,agreement_type,renewal_type,system_status,product_bob,fee_bob,total_bob,unique_products_count,unique_machines_count,unique_machine_variants_count,unique_chemistries_count,unique_branches_count,allied_agreement_count,auto_waste_agreement_count,chemistry_agreement_count,kleenwaste_agreement_count,machine_services_agreement_count,oil_agreement_count,allied_service_interval,auto_waste_service_interval,chemistry_service_interval,kleenwaste_service_interval,machine_services_service_interval,oil_service_interval,allied_total_bob,auto_waste_total_bob,chemistry_total_bob,kleenwaste_total_bob,machine_services_total_bob,oil_total_bob,allied_unit_amount,auto_waste_unit_amount,chemistry_unit_amount,kleenwaste_unit_amount,machine_services_unit_amount,oil_unit_amount,total_cases,total_pull_van,total_new_van,total_van,total_repair_cases,customer_tier,resolution_status,pull_type,case_type,risk_level,case_origin,first_case_creation_date,last_case_creation_date,first_registered_date,last_registered_date,latest_expected_pull_date,churn_label,latest_resolved_date
4308,CGBA200453,9,2022-02-28,2026-02-27,1-9,Exeter,Scheduled Billing,Automatic Renewal,Active,1315.680,173.760,1489.440,4,0,0,0,1,0,4,0,0,0,5,0.000,33.000,0.000,0.000,0.000,14.000,0.000,1082.880,0.000,0.000,0.000,406.560,0.000,22.560,0.000,0.000,0.000,8.470,1,0.000,2638.504,2638.504,0.000,Platinum,Customer Saved,NaN,Risk,Site Access,NaN,NaT,NaT,NaT,NaT,2026-02-27,0,2025-02-11 13:12:09
1530,CGBA015610,6,2021-03-10,2026-03-09,250-499,Bristol,Scheduled Billing,Automatic Renewal,Active,13264.420,2079.840,15344.260,3,2,1,2,1,0,0,0,0,6,0,0.000,0.000,0.000,0.000,4.000,0.000,0.000,0.000,0.000,0.000,15344.260,0.000,0.000,0.000,0.000,0.000,213.115,0.000,1,0.000,15224.984,15224.984,2.000,Diamond,Customer Saved,NaN,Risk,Customer Unsatisfied,NaN,2024-03-14 13:10:00,2024-03-14 13:10:00,2024-03-14 15:10:33,2024-03-14 15:10:33,2025-03-09,0,2024-04-11 08:37:20
4028,CGBA118511,3,2020-09-30,2025-09-29,1-9,West London,Scheduled Billing,Automatic Renewal,Active,1340.280,319.920,1660.200,2,1,1,0,1,0,0,0,0,3,0,0.000,0.000,0.000,0.000,13.000,0.000,0.000,0.000,0.000,0.000,1660.200,0.000,0.000,0.000,0.000,0.000,46.117,0.000,1,0.000,3240.607,3240.607,0.000,Platinum,Customer Saved,NaN,Risk,Site Access,NaN,2023-06-27 06:53:00,2023-06-27 06:53:00,2023-06-27 08:53:52,2023-06-27 08:53:52,2023-09-29,0,2023-07-11 13:49:45
3535,CGBA102590,6,2021-10-31,2025-09-30,50-99,Washington,Scheduled Billing,Automatic Renewal,Active,14212.720,960.000,15172.720,3,2,1,2,1,0,0,0,0,6,0,0.000,0.000,0.000,0.000,5.000,0.000,0.000,0.000,0.000,0.000,15172.720,0.000,0.000,0.000,0.000,0.000,210.732,0.000,1,0.000,14692.720,14692.720,0.000,Diamond,Customer Saved,NaN,Risk,Machine Not Being Used,Notice in Writing,NaT,NaT,NaT,NaT,2020-02-14,0,2021-08-01 04:13:27
2998,CGBA028695,20,2017-07-16,2026-07-15,1-9,Washington,Scheduled Billing,Automatic Renewal,Active,1825.040,302.880,2127.920,8,2,1,2,1,0,6,0,0,6,8,0.000,13.500,0.000,0.000,13.500,33.500,0.000,248.640,0.000,0.000,1796.120,83.160,0.000,3.453,0.000,0.000,24.947,1.155,1,0.000,1921.000,1921.000,0.000,Platinum,Customer Saved,NaN,Risk,Machine Not Being Used,SSR,2018-10-23 08:20:00,2018-10-23 08:20:00,2018-10-23 10:20:53,2018-10-23 10:20:53,2019-11-30,0,2021-10-02 11:57:10


In [175]:
master_df['resolution_status'].value_counts()

resolution_status
Customer Saved    3359
Customer Lost     1263
OPEN              1006
Name: count, dtype: int64

In [173]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5628 entries, 0 to 5627
Data columns (total 59 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   account_number                     5628 non-null   str           
 1   total_agreements                   5628 non-null   int64         
 2   first_agreement_start              5628 non-null   datetime64[us]
 3   last_agreement_end                 5628 non-null   datetime64[us]
 4   company_sizing                     5628 non-null   str           
 5   branch                             5628 non-null   str           
 6   agreement_type                     5628 non-null   str           
 7   renewal_type                       5628 non-null   str           
 8   system_status                      5628 non-null   str           
 9   product_bob                        5628 non-null   float64       
 10  fee_bob                            5628 non-nul

In [176]:
master_df.to_excel('dataset_2/unified_table.xlsx', index=False)